# CiteCheck Quickstart

End-to-end demo of the CiteCheck pipeline using **mocked components** so the notebook is runnable on a fresh checkout with no network, no GPU, no API keys, and no large data downloads.

## What this notebook shows
1. Loading the smoke evaluation fixture (10 synthetic questions)
2. The Bluebook citation grammar in action
3. A mocked `CourtListenerClient` + `CitationResolver`
4. A mocked retriever serving from a small in-memory corpus
5. Running `VanillaBaseline` and `NaiveRAGBaseline` on a few questions
6. Running the full `VerifyLoop` with retraction-or-retry
7. Computing the 7 evaluation metrics on the predictions

## What this notebook does NOT show
- Real LLM inference (we use a deterministic mock generator)
- Real BM25/dense retrieval (we use a Python dict + simple scoring)
- Real NLI scoring (mocked to a fixed value)
- Real CourtListener API calls (mocked to canned responses)
- Real benchmark numbers (use the Phase 2 plan to execute properly)

For real execution see `docs/superpowers/plans/2026-05-24-phd-prep-phase-2-build.md`.

In [ ]:
# Setup
from __future__ import annotations

import json
import logging
from pathlib import Path
from unittest.mock import MagicMock

logging.basicConfig(level=logging.INFO, format='%(name)s %(levelname)s %(message)s')
logger = logging.getLogger('quickstart')

# Locate fixtures relative to the notebook (citecheck/notebooks/ -> citecheck/tests/fixtures/)
NOTEBOOK_DIR = Path('.').resolve()
FIXTURE_DIR = NOTEBOOK_DIR.parent / 'tests' / 'fixtures'
assert (FIXTURE_DIR / 'smoke_eval.jsonl').exists(), f'Missing fixture at {FIXTURE_DIR}'
print(f'Using fixtures from: {FIXTURE_DIR}')

## 1. Load the smoke evaluation set

In [ ]:
from citecheck.data.eval_set import load_eval_set

eval_examples = load_eval_set(FIXTURE_DIR / 'smoke_eval.jsonl')
print(f'Loaded {len(eval_examples)} examples')
for ex in eval_examples[:2]:
    print(f'  {ex.id}: {ex.question[:60]}...')
    print(f'    gold: {ex.gold_citations}')
    print(f'    axis: {ex.metadata.get("axis", "?")}')

## 2. Inspect the Bluebook grammar

In [ ]:
from citecheck.agent import BluebookGrammar

grammar = BluebookGrammar()
test_citations = [
    ('Smith v. Jones, 412 F.3d 567 (9th Cir. 2005)', 'real-looking'),
    ('Brown v. Board of Education, 347 U.S. 483 (1954)', 'real'),
    ('Doe v. Roe, 999 F.4th 1 (12th Cir. 9999)', 'fabricated'),
    ('not a citation', 'malformed'),
]
for cite, kind in test_citations:
    valid = grammar.validate(cite)
    print(f'  [{kind:12s}] valid={valid}  {cite[:60]}')

print(f'\nOutlines regex (first 120 chars): {grammar.as_outlines_regex()[:120]}...')

## 3. Mock CourtListener client + CitationResolver

Returns a canned `CLOpinion` for known smoke-fixture citations; `None` otherwise.

In [ ]:
from citecheck.data.cl_client import CLOpinion
from citecheck.agent import CitationResolver

KNOWN_CITATIONS = {
    'Smith v. Jones, 412 F.3d 567 (9th Cir. 2005)': CLOpinion(
        id=1001, case_name='Smith v. Jones', court='ca9',
        court_jurisdiction='F', year=2005,
        citation_strings=['412 F.3d 567'],
        body_text='A fixture-supplier is liable for design defects when foreseeable.',
        url='https://courtlistener.example/1001'),
    'Celotex Corp. v. Catrett, 477 U.S. 317 (1986)': CLOpinion(
        id=1002, case_name='Celotex Corp. v. Catrett', court='scotus',
        court_jurisdiction='F', year=1986,
        citation_strings=['477 U.S. 317'],
        body_text='The party seeking summary judgment may demonstrate absence of evidence.',
        url='https://courtlistener.example/1002'),
}

mock_cl = MagicMock(name='CourtListenerClient')
mock_cl.resolve_citation.side_effect = lambda cite: next(
    (op for key, op in KNOWN_CITATIONS.items() if key in cite or any(c in cite for c in op.citation_strings)),
    None,
)

resolver = CitationResolver(cl_client=mock_cl)
# Stub NLI to a fixed value to avoid downloading a real DeBERTa model
if hasattr(resolver, '_nli_predict'):
    resolver._nli_predict = lambda *a, **kw: 0.9

# Verify a real and a fake citation
for cite in ['Smith v. Jones, 412 F.3d 567 (9th Cir. 2005)', 'Doe v. Roe, 999 F.4th 1 (12th Cir. 9999)']:
    r = resolver.verify(citation_str=cite, asserted_claim='Fixture-supplier liability under NY law.')
    print(f'  {cite[:55]:55s} -> {r.status.value}')

## 4. Mock retriever serving from the smoke corpus

In [ ]:
from citecheck.retrieval import RetrievalResult

# Load smoke corpus into memory
corpus = []
with (FIXTURE_DIR / 'smoke_corpus.jsonl').open(encoding='utf-8') as f:
    for line in f:
        corpus.append(json.loads(line))
print(f'Loaded {len(corpus)} passages')

class ToyRetriever:
    '''Tiny TF-ish retriever: rank by # of query tokens appearing in the passage.'''
    def __init__(self, corpus):
        self.corpus = corpus
    def search(self, query: str, top_k: int = 5):
        q_tokens = set(query.lower().split())
        scored = []
        for doc in self.corpus:
            text = doc['contents'].lower()
            score = sum(1 for t in q_tokens if t in text)
            scored.append((doc['id'], score, doc['contents']))
        scored.sort(key=lambda x: x[1], reverse=True)
        return [
            RetrievalResult(doc_id=doc_id, score=float(score), rank=i + 1, passage=text)
            for i, (doc_id, score, text) in enumerate(scored[:top_k])
        ]

retriever = ToyRetriever(corpus)
results = retriever.search('fixture supplier liability', top_k=3)
for r in results:
    print(f'  [{r.rank}] {r.doc_id}  score={r.score:.1f}  {r.passage[:70]}...')

## 5. Mock LLM generator

Returns a deterministic answer per question — sometimes with valid cites, sometimes with fakes.

In [ ]:
def mock_generator(prompt: str) -> str:
    '''Toy generator that returns a fixed answer if a known query appears; else a fabricated cite.'''
    if 'fixture' in prompt.lower():
        return 'Yes, the fixture-supplier is liable. See Smith v. Jones, 412 F.3d 567 (9th Cir. 2005).'
    if 'summary judgment' in prompt.lower():
        return 'The standard is set in Celotex Corp. v. Catrett, 477 U.S. 317 (1986).'
    return 'Per Doe v. Roe, 999 F.4th 1 (12th Cir. 9999), the rule applies.'

for q in [ex.question for ex in eval_examples[:3]]:
    print(f'Q: {q[:60]}...')
    print(f'A: {mock_generator(q)}')
    print()

## 6. Run `VanillaBaseline` (no retrieval)

In [ ]:
from citecheck.baselines import VanillaBaseline

vanilla = VanillaBaseline(generator=mock_generator)
for ex in eval_examples[:3]:
    result = vanilla.answer(ex.question)
    print(f'Q ({ex.id}): {ex.question[:55]}...')
    print(f'A: {result.answer_text[:80]}...')
    print(f'Cites parsed: {result.citations}')
    print()

## 7. Run `NaiveRAGBaseline` (with retrieval)

In [ ]:
from citecheck.baselines import NaiveRAGBaseline

naive_rag = NaiveRAGBaseline(generator=mock_generator, retriever=retriever)
for ex in eval_examples[:3]:
    result = naive_rag.answer(ex.question)
    print(f'Q ({ex.id}): {ex.question[:55]}...')
    print(f'A: {result.answer_text[:80]}...')
    print()

## 8. Run the full `VerifyLoop`

Combines generator + retriever + resolver. Citations that fail verification trigger retraction-or-retry (up to `max_iterations`).

In [ ]:
from citecheck.agent import VerifyLoop

loop = VerifyLoop(
    generator=mock_generator,
    retriever=retriever,
    reranker=None,  # skip reranker in demo
    resolver=resolver,
    max_iterations=2,
    use_constrained_decoding=False,
)
predictions = []
for ex in eval_examples[:5]:
    pred = loop.answer(ex.question)
    pred.question_id = ex.id
    predictions.append(pred)
    statuses = [vr.status.value for vr in pred.verification_results]
    print(f'  {ex.id}: {len(pred.citations)} cite(s), statuses={statuses}')

## 9. Compute the 7 evaluation metrics

In [ ]:
from citecheck.eval import aggregate_metrics

report = aggregate_metrics(predictions, eval_examples[:5], human_ratings=None)
print(f'Sample size: {getattr(report, "sample_size", len(predictions))}')
print(f'Resolution rate: {getattr(report, "resolution_rate", "?")}')
print(f'Fabrication rate: {getattr(report, "fabrication_rate", "?")}')
if hasattr(report, '__dict__'):
    for k, v in report.__dict__.items():
        print(f'  {k}: {v}')

## 10. What this notebook didn't show

- **Real data**: production runs use the Caselaw Access Project bulk corpus (~100 GB) — see `scripts/download_cap.py`
- **Real retrieval**: production uses BM25 (pyserini) + dense (BGE + FAISS) + RRF — see `scripts/build_bm25_index.py`, `build_dense_index.py`
- **Real reranker**: production fine-tunes a cross-encoder with QLoRA — see `scripts/train_reranker.py`
- **Real CourtListener**: register at https://www.courtlistener.com/help/api/ for a free API key, then set `CL_API_KEY` in `.env`
- **Real NLI**: production loads DeBERTa-v3-large-mnli on first `CitationResolver.verify` call (~1.5 GB download)
- **Real LLM**: production runs Llama-3.1-8B-Instruct via QLoRA on a 24GB GPU
- **Real eval set**: production has ~500 expert-annotated questions — see `data/eval_set.py` and `Phase 2 Task 24-26`

For the full execution plan see `docs/superpowers/plans/2026-05-24-phd-prep-phase-2-build.md`.